# 04 - Analyse des resultats et des cas d'echec

In [ ]:
import json
import sys
from collections import defaultdict
from pathlib import Path

_c = Path.cwd()
while _c != _c.parent and not (_c / "data" / "raw").is_dir():
    _c = _c.parent
import os
os.chdir(_c)
sys.path.insert(0, str(_c))

In [ ]:
res = json.loads(
    Path("data/evaluation/retrieval_results.json").read_text(encoding="utf-8")
)
print("Agregats :", res["aggregates"])

In [ ]:
qs = json.loads(
    Path("data/evaluation/test_questions.json").read_text(encoding="utf-8")
)["questions"]
qmap = {q["id"]: q for q in qs}
agg = defaultdict(lambda: defaultdict(list))
for r in res["per_question"]:
    c = qmap[r["id"]].get("category", "?")
    agg[c]["p1"].append(r["precision_at_1"])
    agg[c]["h5"].append(r["source_hit_at_5"])
    agg[c]["mrr"].append(r["mrr"])
for c in sorted(agg):
    a = agg[c]
    n = len(a["p1"])
    print(
        f"{c}: n={n} P1={sum(a['p1']) / n:.3f} hit5={sum(a['h5']) / n:.3f} MRR={sum(a['mrr']) / n:.3f}"
    )

In [ ]:
print(
    "== 9 cas d echec du perimetre (source attendue absente des 5 premiers passages) =="
)
for r in res["per_question"]:
    q = qmap.get(r["id"], {})
    if q.get("category") != "hors_perimetre" and r["source_hit_at_5"] == 0:
        print(
            r["id"],
            q.get("category"),
            "-> attendu",
            r["expected_source"],
            "::",
            q.get("question"),
        )

In [ ]:
if Path("data/evaluation/chunking_experiment.json").exists():
    ch = json.loads(
        Path("data/evaluation/chunking_experiment.json").read_text(encoding="utf-8")
    )
    for cfg, v in ch.items():
        print(cfg, "->", v)